# 00: Data Preparation and Validation

This notebook prepares the raw Football-Data CSV files for the analytical pipeline. The goal is to create one clean, reproducible match-level dataset while keeping the most important quality checks visible.

### What this notebook does

- finds the 35 source files across five leagues and seven seasons;
- checks required columns, league labels, dates, duplicates, results, odds coverage, and match-stat coverage;
- standardizes the selected fields into a consistent schema;
- creates a deterministic `match_id`;
- exports the canonical match table and a file-level audit.

### Outputs

- `data/processed/matches_clean.csv`
- `reports/notebook_file_audit.csv`

Run the notebook from top to bottom. The assertions are part of the data-quality process, so a failed assertion should be investigated rather than removed.


## 1. Import libraries

`Path` handles project paths, `pandas` is used for tabular data, and `numpy` is used for a small number of validation operations.


In [1]:
from pathlib import Path

import numpy as np
import pandas as pd

pd.set_option("display.max_columns", 100)

## 2. Resolve project paths

The notebook can be launched from the repository root or from the `notebooks` folder. This cell resolves the project root and defines the raw-data, processed-data, and report directories in one place.


In [2]:
current_dir = Path.cwd().resolve()

PROJECT_ROOT = current_dir.parent if current_dir.name == "notebooks" else current_dir
RAW_DIR = PROJECT_ROOT / "data" / "raw"
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
REPORTS_DIR = PROJECT_ROOT / "reports"

## 3. Find the source CSV files

The raw archive is stored in season subfolders, so `rglob("*.csv")` searches recursively. The expected input is 35 files: five leagues across seven seasons.


In [3]:
EXPECTED_FILE_COUNT = 35

csv_files = sorted(RAW_DIR.rglob("*.csv"))

print(f"Found {len(csv_files)} CSV files")

assert len(csv_files) == EXPECTED_FILE_COUNT, (
    f"Expected {EXPECTED_FILE_COUNT} files, "
    f"found {len(csv_files)}."
)

csv_files[:5]


Found 35 CSV files


[WindowsPath('C:/Users/Admin/Desktop/Projects_Github/football-betting-market-efficiency/data/raw/2019_20/D7.csv'),
 WindowsPath('C:/Users/Admin/Desktop/Projects_Github/football-betting-market-efficiency/data/raw/2019_20/E6.csv'),
 WindowsPath('C:/Users/Admin/Desktop/Projects_Github/football-betting-market-efficiency/data/raw/2019_20/F7.csv'),
 WindowsPath('C:/Users/Admin/Desktop/Projects_Github/football-betting-market-efficiency/data/raw/2019_20/I7.csv'),
 WindowsPath('C:/Users/Admin/Desktop/Projects_Github/football-betting-market-efficiency/data/raw/2019_20/SP7.csv')]

## 4. Inspect one source file

Before combining the archive, inspect one file to confirm that the delimiter, encoding, shape, and basic column structure look reasonable. This is a simple sanity check before the full ingestion step.


In [4]:
sample_path = csv_files[0]
sample = pd.read_csv(sample_path, encoding="utf-8-sig")

print("File:", sample_path)
print("Shape:", sample.shape)
display(sample.head())
print(sample.columns.tolist())
print("League code:", sample["Div"].unique())

File: C:\Users\Admin\Desktop\Projects_Github\football-betting-market-efficiency\data\raw\2019_20\D7.csv
Shape: (306, 105)


,Div,Date,Time,HomeTeam,AwayTeam,FTHG,FTAG,FTR,HTHG,HTAG,HTR,HS,AS,HST,AST,HF,AF,HC,AC,HY,AY,HR,AR,B365H,B365D,B365A,BWH,BWD,BWA,IWH,IWD,IWA,PSH,PSD,PSA,WHH,WHD,WHA,VCH,VCD,VCA,MaxH,MaxD,MaxA,AvgH,AvgD,AvgA,B365>2.5,B365<2.5,P>2.5,...,AHh,B365AHH,B365AHA,PAHH,PAHA,MaxAHH,MaxAHA,AvgAHH,AvgAHA,B365CH,B365CD,B365CA,BWCH,BWCD,BWCA,IWCH,IWCD,IWCA,PSCH,PSCD,PSCA,WHCH,WHCD,WHCA,VCCH,VCCD,VCCA,MaxCH,MaxCD,MaxCA,AvgCH,AvgCD,AvgCA,B365C>2.5,B365C<2.5,PC>2.5,PC<2.5,MaxC>2.5,MaxC<2.5,AvgC>2.5,AvgC<2.5,AHCh,B365CAHH,B365CAHA,PCAHH,PCAHA,MaxCAHH,MaxCAHA,AvgCAHH,AvgCAHA
0,D1,16/08/2019,19:30,Bayern Munich,Hertha,2,2,D,1,2,A,17,6,7,3,6,17,12,0,3,3,0,0,1.14,8.00,15.00,1.18,7.0,16.0,1.25,6.1,11.00,1.19,7.73,15.30,1.18,7.5,15.00,1.18,8.0,15.0,1.25,8.30,17.50,1.18,7.55,15.04,1.33,3.40,1.36,...,-2.00,1.80,2.00,1.85,2.07,2.10,2.11,1.84,2.04,1.14,8.0,15.0,1.18,8.00,13.00,1.25,6.1,11.0,1.18,8.49,17.38,1.15,8.5,17.00,1.17,8.0,18.0,1.25,9.10,19.25,1.17,8.00,15.67,1.30,3.50,1.32,3.62,1.40,3.62,1.31,3.40,-2.25,2.03,1.90,1.99,1.93,2.04,1.93,1.98,1.91
1,D1,17/08/2019,14:30,Dortmund,Augsburg,5,1,H,1,1,D,23,5,10,1,8,6,10,0,0,0,0,0,1.20,7.00,13.00,1.22,6.5,12.5,1.25,6.1,11.00,1.23,6.76,13.52,1.20,6.5,15.00,1.22,6.5,15.0,1.25,7.15,17.00,1.22,6.62,13.38,1.44,2.75,1.44,...,-2.00,2.02,1.77,2.03,1.88,2.10,1.90,2.02,1.85,1.12,8.0,26.0,1.16,8.00,16.50,1.20,6.5,15.0,1.16,8.37,20.11,1.20,6.5,15.00,1.15,8.5,20.0,1.20,8.80,26.00,1.16,8.10,18.29,1.30,3.50,1.33,3.50,1.40,3.60,1.32,3.31,-2.25,1.92,2.01,1.92,2.00,1.98,2.04,1.91,1.97
2,D1,17/08/2019,14:30,Freiburg,Mainz,3,0,H,0,0,D,19,19,8,5,6,15,5,5,1,3,0,0,2.25,3.25,3.40,2.20,3.3,3.4,2.15,3.4,3.35,2.23,3.45,3.44,2.20,3.3,3.40,2.20,3.4,3.4,2.26,3.49,3.65,2.20,3.37,3.36,1.90,1.90,1.95,...,-0.25,1.94,1.99,1.93,2.00,1.94,2.01,1.91,1.98,2.55,3.3,2.7,2.50,3.30,2.85,2.60,3.3,2.7,2.74,3.30,2.77,2.20,3.3,3.40,2.63,3.3,2.8,2.74,3.38,2.96,2.61,3.29,2.78,1.90,1.90,1.98,1.92,2.01,1.96,1.93,1.89,0.00,1.92,2.01,1.94,1.97,1.97,2.06,1.90,1.99
3,D1,17/08/2019,14:30,Leverkusen,Paderborn,3,2,H,2,2,D,13,11,4,6,8,9,6,6,2,0,0,0,1.25,6.00,12.00,1.30,5.5,10.0,1.27,5.8,10.50,1.29,6.07,10.57,1.27,5.8,11.00,1.29,6.0,10.5,1.31,6.40,12.25,1.28,5.97,10.27,1.36,3.20,1.39,...,-1.75,1.98,1.95,1.97,1.94,1.99,1.97,1.95,1.93,1.22,6.5,11.0,1.25,6.25,11.00,1.27,5.8,10.5,1.27,6.69,10.45,1.25,6.0,12.00,1.25,6.5,11.0,1.29,7.05,13.00,1.25,6.52,10.70,1.28,3.75,1.30,3.73,1.35,3.85,1.29,3.58,-2.00,2.07,1.86,2.05,1.86,2.15,1.91,2.03,1.85
4,D1,17/08/2019,14:30,Werder Bremen,Fortuna Dusseldorf,1,3,A,0,1,A,23,12,10,7,8,13,14,5,0,2,0,0,1.75,3.75,4.75,1.75,4.0,4.4,1.70,4.0,4.70,1.75,4.05,4.68,1.73,3.9,4.75,1.75,4.0,4.6,1.80,4.05,4.95,1.74,3.93,4.57,1.66,2.20,1.72,...,-0.75,1.99,1.94,1.99,1.93,1.99,1.95,1.96,1.92,1.66,4.2,4.5,1.70,4.00,4.75,1.65,4.1,5.0,1.70,4.15,4.90,1.70,4.0,4.75,1.70,4.2,4.8,1.76,4.32,5.10,1.69,4.07,4.79,1.66,2.20,1.68,2.30,1.69,2.38,1.65,2.26,-0.75,1.92,2.01,1.92,2.00,1.95,2.11,1.89,2.00


['Div', 'Date', 'Time', 'HomeTeam', 'AwayTeam', 'FTHG', 'FTAG', 'FTR', 'HTHG', 'HTAG', 'HTR', 'HS', 'AS', 'HST', 'AST', 'HF', 'AF', 'HC', 'AC', 'HY', 'AY', 'HR', 'AR', 'B365H', 'B365D', 'B365A', 'BWH', 'BWD', 'BWA', 'IWH', 'IWD', 'IWA', 'PSH', 'PSD', 'PSA', 'WHH', 'WHD', 'WHA', 'VCH', 'VCD', 'VCA', 'MaxH', 'MaxD', 'MaxA', 'AvgH', 'AvgD', 'AvgA', 'B365>2.5', 'B365<2.5', 'P>2.5', 'P<2.5', 'Max>2.5', 'Max<2.5', 'Avg>2.5', 'Avg<2.5', 'AHh', 'B365AHH', 'B365AHA', 'PAHH', 'PAHA', 'MaxAHH', 'MaxAHA', 'AvgAHH', 'AvgAHA', 'B365CH', 'B365CD', 'B365CA', 'BWCH', 'BWCD', 'BWCA', 'IWCH', 'IWCD', 'IWCA', 'PSCH', 'PSCD', 'PSCA', 'WHCH', 'WHCD', 'WHCA', 'VCCH', 'VCCD', 'VCCA', 'MaxCH', 'MaxCD', 'MaxCA', 'AvgCH', 'AvgCD', 'AvgCA', 'B365C>2.5', 'B365C<2.5', 'PC>2.5', 'PC<2.5', 'MaxC>2.5', 'MaxC<2.5', 'AvgC>2.5', 'AvgC<2.5', 'AHCh', 'B365CAHH', 'B365CAHA', 'PCAHH', 'PCAHA', 'MaxCAHH', 'MaxCAHA', 'AvgCAHH', 'AvgCAHA']
League code: ['D1']


## 5. Load all files and build the league-season audit

Each CSV is read separately and checked before it is added to the combined dataset. The season comes from the folder structure, while the league is taken from the internal `Div` field rather than the downloaded filename.

The audit also records row counts, source schemas, and coverage of the closing-odds fields used later in the project.


In [5]:
CORE_COLUMNS = [
    "Div", "Date", "HomeTeam", "AwayTeam",
    "FTHG", "FTAG", "FTR"
]

ODDS_GROUPS = {
    "avg_close_coverage": ["AvgCH", "AvgCD", "AvgCA"],
    "b365_close_coverage": ["B365CH", "B365CD", "B365CA"],
}

META_COLUMNS = ["season", "source_file"]


def calculate_coverage(df, columns):
    return df.reindex(columns=columns).notna().all(axis=1).mean()


frames = []
file_checks = []

for path in csv_files:
    temp = pd.read_csv(path, encoding="utf-8-sig")

    missing_core = sorted(set(CORE_COLUMNS) - set(temp.columns))

    if missing_core:
        raise ValueError(f"{path.name}: missing required columns {missing_core}")

    league_values = temp["Div"].dropna().unique()

    if len(league_values) != 1:
        raise ValueError(
            f"{path.name}: expected one Div value, "
            f"found {league_values}"
        )

    season = path.parent.name

    temp = temp.assign(
        season=season,
        source_file=path.name
    )

    source_columns = temp.columns.drop(META_COLUMNS)

    odds_coverage = {
        name: calculate_coverage(temp, columns)
        for name, columns in ODDS_GROUPS.items()
    }

    file_checks.append({
        "season": season,
        "league_code": league_values[0],
        "source_file": path.name,
        "rows": len(temp),
        "columns": len(source_columns),
        "schema_signature": "|".join(source_columns),
        **odds_coverage,
    })

    frames.append(temp)


file_audit = (
    pd.DataFrame(file_checks)
    .sort_values(["season", "league_code"])
)

display(file_audit.drop(columns="schema_signature"))

print(
    "Distinct source schemas:",
    file_audit["schema_signature"].nunique()
)

,season,league_code,source_file,rows,columns,avg_close_coverage,b365_close_coverage
0,2019_20,D1,D7.csv,306,105,1.0,1.000000
1,2019_20,E0,E6.csv,380,106,1.0,1.000000
2,2019_20,F1,F7.csv,279,105,1.0,1.000000
3,2019_20,I1,I7.csv,380,105,1.0,1.000000
4,2019_20,SP1,SP7.csv,380,105,1.0,1.000000
5,2020_21,D1,D6.csv,306,105,1.0,1.000000
6,2020_21,E0,E5.csv,380,106,1.0,1.000000
7,2020_21,F1,F6.csv,380,105,1.0,0.997368
8,2020_21,I1,I6.csv,380,105,1.0,1.000000
9,2020_21,SP1,SP6.csv,380,105,1.0,1.000000


Distinct source schemas: 6


## 6. Combine the source tables

`pd.concat()` stacks all league-season files into one DataFrame. Historical files do not always share the same full schema, so columns that are absent in a particular file are kept as missing values rather than forcing positional alignment.


In [6]:

df_raw = pd.concat(frames, ignore_index=True, sort=False).copy()

print("Combined shape:", df_raw.shape)
print(sorted(df_raw["season"].unique()))
print(sorted(df_raw["Div"].unique()))

core_missing_values = df_raw[CORE_COLUMNS].isna().sum()

print("Missing values in core columns:")
display(core_missing_values)

assert core_missing_values.sum() == 0
assert len(df_raw) == 12_459

# df_raw

Combined shape: (12459, 164)
['2019_20', '2020_21', '2021_22', '2022_23', '2023_24', '2024_25', '2025_26']
['D1', 'E0', 'F1', 'I1', 'SP1']
Missing values in core columns:


Div         0
Date        0
HomeTeam    0
AwayTeam    0
FTHG        0
FTAG        0
FTR         0
dtype: int64

## 7. Validate and convert match dates

Dates are converted with `errors="coerce"` so malformed values become `NaT` and can be inspected explicitly. The notebook requires every match date to convert successfully.


In [7]:
df_raw["match_date"] = pd.to_datetime(df_raw["Date"], dayfirst=True, errors="coerce")
bad_dates = df_raw[df_raw["match_date"].isna()]

print("Invalid dates:", len(bad_dates))
display(bad_dates[["season", "Div", "Date", "HomeTeam", "AwayTeam"]].head())
assert bad_dates.empty, "Some match dates could not be converted."

Invalid dates: 0


,season,Div,Date,HomeTeam,AwayTeam


## 8. Check for duplicate matches

A match is identified by season, league, date, home team, and away team. Using `keep=False` would expose every row involved in a duplicated key, making any collision easy to inspect.


In [8]:
MATCH_KEY = ["season", "Div", "match_date", "HomeTeam", "AwayTeam"]
duplicate_matches = df_raw[df_raw.duplicated(MATCH_KEY, keep=False)]

print("Rows involved in duplicate match keys:", len(duplicate_matches))
display(duplicate_matches[MATCH_KEY].head())
assert duplicate_matches.empty, "Duplicate matches detected."

Rows involved in duplicate match keys: 0


,season,Div,match_date,HomeTeam,AwayTeam


## 9. Reconcile the recorded result with the score

The expected full-time result is reconstructed from home and away goals and compared with `FTR`. This protects downstream outcome labels from silent source inconsistencies.


In [9]:
df_raw["expected_ftr"] = np.select(
    [df_raw["FTHG"] > df_raw["FTAG"], df_raw["FTHG"] == df_raw["FTAG"]],
    ["H", "D"],
    default="A",
)

result_mismatches = df_raw[df_raw["FTR"] != df_raw["expected_ftr"]]
print("Result and score mismatches:", len(result_mismatches))
display(result_mismatches[["season", "Div", "HomeTeam", "AwayTeam", "FTHG", "FTAG", "FTR", "expected_ftr"]].head())
assert result_mismatches.empty, "FTR does not match the scoreline."

Result and score mismatches: 0


,season,Div,HomeTeam,AwayTeam,FTHG,FTAG,FTR,expected_ftr


## 10. Validate closing-odds coverage

Calibration and ROI use different closing-odds sources. Average-market closing odds are required for the calibration analysis, while Bet365 closing odds are required for the bookmaker ROI analysis.

A missing Bet365 market therefore affects only the analyses that depend on Bet365 prices; it does not invalidate the match for every part of the project.


In [10]:
AVG_CLOSE_COLUMNS = ["AvgCH", "AvgCD", "AvgCA"]
B365_CLOSE_COLUMNS = ["B365CH", "B365CD", "B365CA"]

avg_close_missing_mask = (
    df_raw[AVG_CLOSE_COLUMNS].isna().any(axis=1)
)

avg_close_invalid_mask = (
    (df_raw[AVG_CLOSE_COLUMNS] <= 1).any(axis=1)
)

b365_close_missing_mask = (
    df_raw[B365_CLOSE_COLUMNS].isna().any(axis=1)
)

b365_complete_mask = ~b365_close_missing_mask

b365_close_invalid_mask = (
    b365_complete_mask
    & (df_raw[B365_CLOSE_COLUMNS] <= 1).any(axis=1)
)

avg_close_missing = df_raw[avg_close_missing_mask]
avg_close_invalid = df_raw[avg_close_invalid_mask]
b365_close_missing = df_raw[b365_close_missing_mask]
b365_close_invalid = df_raw[b365_close_invalid_mask]

print("Missing AvgC markets:", len(avg_close_missing))
print("Invalid AvgC markets:", len(avg_close_invalid))
print("Missing B365C markets:", len(b365_close_missing))
print("Invalid B365C markets:", len(b365_close_invalid))

display(
    b365_close_missing[
        [
            "season",
            "Div",
            "match_date",
            "HomeTeam",
            "AwayTeam",
            *B365_CLOSE_COLUMNS,
        ]
    ]
)

assert avg_close_missing.empty, "Missing AvgC closing odds detected."
assert avg_close_invalid.empty, "AvgC closing odds <= 1 detected."
assert b365_close_invalid.empty, "Available B365C closing odds <= 1 detected."
assert len(b365_close_missing) == 1, (
    f"Expected 1 match with an incomplete B365C trio, "
    f"found {len(b365_close_missing)}."
)

Missing AvgC markets: 0
Invalid AvgC markets: 0
Missing B365C markets: 1
Invalid B365C markets: 0


,season,Div,match_date,HomeTeam,AwayTeam,B365CH,B365CD,B365CA
2477,2020_21,F1,2020-10-18,Monaco,Montpellier,NaN,NaN,NaN


## 11. Audit match-stat coverage

Missing match statistics do not automatically remove a match from the canonical dataset. This check identifies incomplete rows so later analyses can apply field-specific exclusions only when those statistics are actually required.


In [11]:
MATCH_STATS = ["HS", "AS", "HST", "AST", "HF", "AF", "HC", "AC", "HY", "AY", "HR", "AR"]
missing_match_stats = df_raw[df_raw[MATCH_STATS].isna().any(axis=1)]
missing_match_stats[MATCH_STATS].isna().sum(axis=1)
print("Matches with at least one missing match statistic:", len(missing_match_stats))
display(missing_match_stats[["season", "Div", "Date", "HomeTeam", "AwayTeam"] + MATCH_STATS])
display(missing_match_stats[MATCH_STATS].isna().sum(axis=1))

Matches with at least one missing match statistic: 1


,season,Div,Date,HomeTeam,AwayTeam,HS,AS,HST,AST,HF,AF,HC,AC,HY,AY,HR,AR
9076,2024_25,D1,14/12/2024,Union Berlin,Bochum,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


9076    12
dtype: int64

## 12. Select and standardize the canonical columns

The raw files contain far more fields than the project needs. This step keeps the documented analytical subset, converts names to `snake_case`, and creates a deterministic `match_id` from the match identity fields.


In [12]:
COLUMN_MAP = {
    "season": "season", "source_file": "source_file", "Div": "div",
    "Date": "match_date_source", "Time": "kickoff_time",
    "HomeTeam": "home_team", "AwayTeam": "away_team",
    "FTHG": "fthg", "FTAG": "ftag", "FTR": "ftr",
    "HS": "home_shots", "AS": "away_shots",
    "HST": "home_shots_on_target", "AST": "away_shots_on_target",
    "HF": "home_fouls", "AF": "away_fouls",
    "HC": "home_corners", "AC": "away_corners",
    "HY": "home_yellow_cards", "AY": "away_yellow_cards",
    "HR": "home_red_cards", "AR": "away_red_cards",
    "AvgH": "avg_h", "AvgD": "avg_d", "AvgA": "avg_a",
    "AvgCH": "avg_ch", "AvgCD": "avg_cd", "AvgCA": "avg_ca",
    "B365H": "b365_h", "B365D": "b365_d", "B365A": "b365_a",
    "B365CH": "b365_ch", "B365CD": "b365_cd", "B365CA": "b365_ca",
    "MaxCH": "max_ch", "MaxCD": "max_cd", "MaxCA": "max_ca",
}

df_clean = (
    df_raw
    .reindex(columns=COLUMN_MAP.keys())
    .rename(columns=COLUMN_MAP)
    .copy()
)

df_clean["match_date"] = pd.to_datetime(
    df_clean.pop("match_date_source"),
    dayfirst=True
)

df_clean["match_id"] = (
    df_clean["season"].astype(str)
    + "_" + df_clean["div"].astype(str)
    + "_" + df_clean["match_date"].dt.strftime("%Y%m%d")
    + "_" + df_clean["home_team"].astype(str)
    + "_" + df_clean["away_team"].astype(str)
)

df_clean.insert(0, "match_id", df_clean.pop("match_id"))

print("Canonical table shape:", df_clean.shape)
display(df_clean.head())

assert df_clean["match_id"].is_unique, "match_id is not unique."

Canonical table shape: (12459, 38)


,match_id,season,source_file,div,kickoff_time,home_team,away_team,fthg,ftag,ftr,home_shots,away_shots,home_shots_on_target,away_shots_on_target,home_fouls,away_fouls,home_corners,away_corners,home_yellow_cards,away_yellow_cards,home_red_cards,away_red_cards,avg_h,avg_d,avg_a,avg_ch,avg_cd,avg_ca,b365_h,b365_d,b365_a,b365_ch,b365_cd,b365_ca,max_ch,max_cd,max_ca,match_date
0,2019_20_D1_20190816_Bayern Munich_Hertha,2019_20,D7.csv,D1,19:30,Bayern Munich,Hertha,2,2,D,17.0,6.0,7.0,3.0,6.0,17.0,12.0,0.0,3.0,3.0,0.0,0.0,1.18,7.55,15.04,1.17,8.00,15.67,1.14,8.00,15.00,1.14,8.0,15.0,1.25,9.10,19.25,2019-08-16
1,2019_20_D1_20190817_Dortmund_Augsburg,2019_20,D7.csv,D1,14:30,Dortmund,Augsburg,5,1,H,23.0,5.0,10.0,1.0,8.0,6.0,10.0,0.0,0.0,0.0,0.0,0.0,1.22,6.62,13.38,1.16,8.10,18.29,1.20,7.00,13.00,1.12,8.0,26.0,1.20,8.80,26.00,2019-08-17
2,2019_20_D1_20190817_Freiburg_Mainz,2019_20,D7.csv,D1,14:30,Freiburg,Mainz,3,0,H,19.0,19.0,8.0,5.0,6.0,15.0,5.0,5.0,1.0,3.0,0.0,0.0,2.20,3.37,3.36,2.61,3.29,2.78,2.25,3.25,3.40,2.55,3.3,2.7,2.74,3.38,2.96,2019-08-17
3,2019_20_D1_20190817_Leverkusen_Paderborn,2019_20,D7.csv,D1,14:30,Leverkusen,Paderborn,3,2,H,13.0,11.0,4.0,6.0,8.0,9.0,6.0,6.0,2.0,0.0,0.0,0.0,1.28,5.97,10.27,1.25,6.52,10.70,1.25,6.00,12.00,1.22,6.5,11.0,1.29,7.05,13.00,2019-08-17
4,2019_20_D1_20190817_Werder Bremen_Fortuna Duss...,2019_20,D7.csv,D1,14:30,Werder Bremen,Fortuna Dusseldorf,1,3,A,23.0,12.0,10.0,7.0,8.0,13.0,14.0,5.0,0.0,2.0,0.0,0.0,1.74,3.93,4.57,1.69,4.07,4.79,1.75,3.75,4.75,1.66,4.2,4.5,1.76,4.32,5.10,2019-08-17


## 13. Export the canonical dataset

After all validation checks pass, the notebook writes the standardized match table and the file-level audit used for project documentation and downstream SQL ingestion.


In [13]:
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
REPORTS_DIR.mkdir(parents=True, exist_ok=True)

matches_output = PROCESSED_DIR / "matches_clean.csv"
audit_output = REPORTS_DIR / "notebook_file_audit.csv"

df_clean.to_csv(matches_output, index=False)
file_audit.to_csv(audit_output, index=False)

print(f"Saved {len(df_clean):,} matches to: {matches_output}")
print(f"Saved the {len(file_audit)}-file audit to: {audit_output}")

Saved 12,459 matches to: C:\Users\Admin\Desktop\Projects_Github\football-betting-market-efficiency\data\processed\matches_clean.csv
Saved the 35-file audit to: C:\Users\Admin\Desktop\Projects_Github\football-betting-market-efficiency\reports\notebook_file_audit.csv
